# Drug Retrieval Evaluation Delivery

这个 notebook 用于交付给其他组，说明我们如何基于 `medical_questions_dataset.json` 评估药物召回效果，以及如何理解向量库的匹配准确性。

> Evaluation scope note:\n
> This evaluation reports label-based retrieval accuracy, not real clinical prescription accuracy.\n
> 也就是说，这里的评估是在看：给定 disease / symptoms labels，我们的系统能否把标签匹配更好的药排在前面；而不是判断药物在真实临床场景下是否一定正确。

## 1. Evaluation Terms

这里保留英文术语，中文解释如下：

- `strict match`: 返回的药物同时命中 query 的 disease label，并且命中至少一个 symptom label。
- `loose match`: 返回的药物只要命中 disease 或 symptom 任一项即可。
- `Hit@K`: Top-K 结果里，是否至少出现一个相关药物。值是 0 或 1，最后对所有 query 取平均。
- `Precision@K`: Top-K 结果中，相关药物所占的比例。
- `MRR` (`Mean Reciprocal Rank`): 第一个相关结果出现得越靠前，分数越高。
- `nDCG` (`Normalized Discounted Cumulative Gain`): 不只看有没有相关结果，也看相关结果是不是排在更靠前的位置。

重点指标解释：

- `hit@5_strict`: Top-5 结果里，是否至少有一个药同时命中正确 disease 和 symptom。
- `precision@5_strict`: Top-5 结果里，有多少比例的药同时命中正确 disease 和 symptom。
- `mrr_strict`: 第一个 strict relevant 药出现在越前面，分数越高；如果总在第 1 名，分数就接近 1。
- `ndcg@5`: 在 Top-5 内综合衡量排序质量，越高表示越相关的药越靠前。
- `ndcg@10`: 与 `ndcg@5` 同理，只是观察窗口扩大到 Top-10。

## 2. Methods Being Compared

- `baseline`: 纯标签匹配，对 `disease_overlap` 和 `symptom_overlap` 做过滤与排序。
- `pure_knn`: 不做 baseline 粗筛，直接用向量库对全药库做相似度检索。这个最能代表向量库本身的匹配准确性。
- `hybrid`: 先用 baseline 粗筛，再用向量相似度做 rerank。这个最能代表向量库在候选集精排里的增益。

In [1]:
from pathlib import Path
import sys
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from app.embedded_module.evaluation import (
    EVALUATION_SCOPE_NOTE,
    default_drug_csv,
    default_embedding_npy,
    default_questions_json,
    run_evaluation,
)

QUESTIONS_JSON = default_questions_json()
DRUG_CSV = default_drug_csv()
EMBEDDING_NPY = default_embedding_npy()

print('QUESTIONS_JSON =', QUESTIONS_JSON)
print('DRUG_CSV =', DRUG_CSV)
print('EMBEDDING_NPY =', EMBEDDING_NPY, '| exists =', EMBEDDING_NPY.exists())
print(EVALUATION_SCOPE_NOTE)

QUESTIONS_JSON = /Users/jayden/Desktop/7012 datamining and text/project_march/ARIN7102_Group_Project/app/dataset_module/bert_training_dataset/medical_questions_dataset.json
DRUG_CSV = /Users/jayden/Desktop/7012 datamining and text/project_march/data/enhanced_drug_table_v1.csv
EMBEDDING_NPY = /Users/jayden/Desktop/7012 datamining and text/project_march/ARIN7102_Group_Project/interaction/drug_comprehensive_embeddings.npy | exists = True
This evaluation reports label-based retrieval accuracy, not real clinical prescription accuracy.


## 3. Run Baseline Evaluation

`baseline` 不依赖 embedding 文件，通常最容易先跑通。

In [2]:
baseline_query_metrics, baseline_ranked_results, baseline_summary = run_evaluation(
    method='baseline',
    questions_json=QUESTIONS_JSON,
    drug_csv=DRUG_CSV,
    limit=8000,
)
baseline_summary

,metric,value
0,need_first_aid,0.263625
1,retrieved_count,9.969125
2,relevant_loose_total,1144.565375
3,relevant_strict_total,170.060125
4,hit@1_loose,1.000000
5,hit@1_strict,1.000000
6,precision@1_loose,1.000000
7,precision@1_strict,1.000000
8,recall@1_loose,0.003912
9,recall@1_strict,0.024751


## 4. Run Pure KNN Evaluation

`pure_knn` 最能代表向量库本身的匹配准确性。\n
如果这里报错，通常是因为 embedding 文件路径不对，或者本地模型 / 依赖没有准备好。

In [3]:
pure_knn_query_metrics, pure_knn_ranked_results, pure_knn_summary = run_evaluation(
    method='pure_knn',
    questions_json=QUESTIONS_JSON,
    drug_csv=DRUG_CSV,
    embedding_npy=EMBEDDING_NPY,
    limit=100,
)
pure_knn_summary

[DrugEmbeddingEngine] 设备: mps
[DrugEmbeddingEngine] 正在加载模型: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[DrugEmbeddingEngine] 模型加载完成


Encoding drugs: 100%|██████████| 1/1 [00:00<00:00,  4.12it/s]


...

Encoding drugs: 100%|██████████| 1/1 [00:00<00:00, 42.59it/s]


[DrugEmbeddingEngine] 编码完成: shape = (1, 768)


Encoding drugs: 100%|██████████| 1/1 [00:00<00:00, 71.76it/s]


[DrugEmbeddingEngine] 编码完成: shape = (1, 768)


,metric,value
0,need_first_aid,0.260000
1,retrieved_count,10.000000
2,relevant_loose_total,791.800000
3,relevant_strict_total,330.200000
4,hit@1_loose,0.620000
5,hit@1_strict,0.600000
6,precision@1_loose,0.620000
7,precision@1_strict,0.600000
8,recall@1_loose,0.001466
9,recall@1_strict,0.001966


## 5. Run Hybrid Evaluation

`hybrid` 先做 baseline 粗筛，再用向量分数做精排。\n
如果 `pure_knn` 一般但 `hybrid` 更好，通常说明向量库更适合做 reranker，而不是单独全库召回。

In [4]:
hybrid_query_metrics, hybrid_ranked_results, hybrid_summary = run_evaluation(
    method='hybrid',
    questions_json=QUESTIONS_JSON,
    drug_csv=DRUG_CSV,
    embedding_npy=EMBEDDING_NPY,
    limit=100,
)
hybrid_summary

[DrugEmbeddingEngine] 设备: mps
[DrugEmbeddingEngine] 正在加载模型: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[DrugEmbeddingEngine] 模型加载完成


Encoding drugs: 100%|██████████| 1/1 [00:00<00:00, 45.34it/s]

...

Encoding drugs: 100%|██████████| 1/1 [00:00<00:00, 21.47it/s]


[DrugEmbeddingEngine] 编码完成: shape = (1, 768)


Encoding drugs: 100%|██████████| 1/1 [00:00<00:00, 47.03it/s]


[DrugEmbeddingEngine] 编码完成: shape = (1, 768)


,metric,value
0,need_first_aid,0.260000
1,retrieved_count,10.000000
2,relevant_loose_total,791.800000
3,relevant_strict_total,330.200000
4,hit@1_loose,1.000000
5,hit@1_strict,1.000000
6,precision@1_loose,1.000000
7,precision@1_strict,1.000000
8,recall@1_loose,0.002222
9,recall@1_strict,0.004032


## 6. Compare Methods

下面这张表是最适合对外展示的 summary table。

In [5]:
def summary_to_row(summary_df, method_name):
    metric_map = dict(zip(summary_df['metric'], summary_df['value']))
    keep = [
        'hit@1_strict',
        'hit@3_strict',
        'hit@5_strict',
        'precision@5_strict',
        'mrr_strict',
        'ndcg@5',
        'ndcg@10',
    ]
    row = {'method': method_name}
    for key in keep:
        row[key] = metric_map.get(key)
    return row

comparison_df = pd.DataFrame([
    summary_to_row(baseline_summary, 'baseline'),
    summary_to_row(pure_knn_summary, 'pure_knn'),
    summary_to_row(hybrid_summary, 'hybrid'),
])
comparison_df

,method,hit@1_strict,hit@3_strict,hit@5_strict,precision@5_strict,mrr_strict,ndcg@5,ndcg@10
0,baseline,1.0,1.00,1.0,0.991675,1.000000,1.000000,0.999996
1,pure_knn,0.6,0.66,0.7,0.570000,0.638429,0.591363,0.587502
2,hybrid,1.0,1.00,1.0,1.000000,1.000000,1.000000,1.000000


## 7. How To Interpret the Results

- 如果 `baseline` 很高，这是正常的，因为它和标签匹配规则天然一致。
- 如果 `pure_knn` 也高，说明向量库本身对 disease / symptoms query 有较强匹配能力。
- 如果 `pure_knn` 一般，但 `hybrid` 明显更好，说明向量库更适合做 reranking。
- 如果 `hybrid` 和 `baseline` 差不多，说明当前 embedding 增益有限，baseline 已经解释了大部分结果。

## 8. Optional Case Study

如果需要对外展示具体 query 的结果，可以查看某个 query 在三种方法下的排序差异。

In [6]:
sample_question_id = 0

print('Baseline')
display(baseline_ranked_results[baseline_ranked_results['question_id'] == sample_question_id].head(5))

print('Pure KNN')
display(pure_knn_ranked_results[pure_knn_ranked_results['question_id'] == sample_question_id].head(5))

print('Hybrid')
display(hybrid_ranked_results[hybrid_ranked_results['question_id'] == sample_question_id].head(5))

Baseline


,question_id,method,rank,drug_name,eval_disease_hit,eval_symptom_hit,eval_loose_hit,eval_strict_hit,eval_relevance,knn_score,avg_rating
0,0,baseline,1,absorbine jr.,1,1,1,1,2,NaN,10.0
1,0,baseline,2,auranofin,1,1,1,1,2,NaN,10.0
2,0,baseline,3,cefotaxime,1,1,1,1,2,NaN,10.0
3,0,baseline,4,flanax pain reliever,1,1,1,1,2,NaN,10.0
4,0,baseline,5,glucosamine,1,1,1,1,2,NaN,10.0


Pure KNN


,question_id,method,rank,drug_name,eval_disease_hit,eval_symptom_hit,eval_loose_hit,eval_strict_hit,eval_relevance,knn_score,avg_rating
0,0,pure_knn,1,voltaren gel,1,1,1,1,2,0.969527,7.99
1,0,pure_knn,2,s-adenosylmethionine,1,1,1,1,2,0.966000,7.71
2,0,pure_knn,3,prednicot,1,1,1,1,2,0.965751,10.00
3,0,pure_knn,4,rheumatrex dose pack,1,1,1,1,2,0.965720,2.00
4,0,pure_knn,5,absorbine jr.,1,1,1,1,2,0.964652,10.00


Hybrid


,question_id,method,rank,drug_name,eval_disease_hit,eval_symptom_hit,eval_loose_hit,eval_strict_hit,eval_relevance,knn_score,avg_rating
0,0,hybrid,1,voltaren gel,1,1,1,1,2,0.969527,7.99
1,0,hybrid,2,s-adenosylmethionine,1,1,1,1,2,0.966000,7.71
2,0,hybrid,3,prednicot,1,1,1,1,2,0.965751,10.00
3,0,hybrid,4,rheumatrex dose pack,1,1,1,1,2,0.965720,2.00
4,0,hybrid,5,absorbine jr.,1,1,1,1,2,0.964652,10.00
